# Composable Conditional Risk in Caller-Owned Models

This walkthrough demonstrates the Feature 016 composition boundary using reviewed, immutable OEL fixtures. It does not fit an SPN, compile a new circuit, regenerate evidence, enumerate decisions, or optimize a benchmark. Instead, it reloads the exact current-SPFlow artifact and max/PWL conditional pairs already validated by automated tests. Its main question is whether a caller can reuse those pairs inside its own Pyomo model while retaining the intended conditional-risk meaning.

The event is violation $Y=1$. For retained decisions $X=x$, OEL exposes two compiled roots:

$$z_{\mathrm{num}}(x) \approx \log p_{X,Y}(x,1),$$

$$z_{\mathrm{den}}(x) \approx \log p_X(x).$$

For continuous $x$, these are **continuous densities, not point probabilities**. Their difference is the authoritative conditional log-risk,

$$z_{\mathrm{num}}(x)-z_{\mathrm{den}}(x)\approx \log P(Y=1\mid X=x),$$

The raw $z_{\mathrm{num}}(x)-z_{\mathrm{den}}(x)$ is retained without clamping, and exponentiating it gives a diagnostic conditional probability. If the displayed log-risk is $L$, read the corresponding probability as $\exp(L)$. The denominator is an **explicit learned denominator** from the same fitted SPN, even when the intended decision-sampling design was uniform.

A successful run establishes composition and implementation parity. It does **not** establish that either compiler approximation is scientifically adequate, that a chance constraint is feasible, or that a displayed decision is optimal. Those questions require separate evidence or a caller-owned optimization.

## Ownership boundary

The **caller owns** the primary decision variables, objective, deterministic constraints, $\alpha$, chance constraint, and decision to invoke a solver. **OEL owns** only the namespaced attachment block: the numerator and denominator circuits, auxiliary variables, decision-coordinate links, validity constraints, and categorical exactly-one constraint.

The fixed-assignment examples below are objective-free diagnostic solves. The final policy example only composes a caller objective and constraints around the returned expression; it is intentionally left unsolved.

### What embedding means here

Embedding is the translation of an already-compiled conditional pair into Pyomo variables, expressions, and constraints inside an existing caller model. It is not a statistical feature embedding. The attached block references caller decisions and exposes the two roots and their log-risk difference; the surrounding optimization remains caller-owned.

![Conditional MIO embedding ownership flow](conditional_mio_embedding_ownership.svg)

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import json
import math
from pathlib import Path

import highspy
import matplotlib.pyplot as plt
import pyomo.environ as pyo

from open_energy_lab.tpm import (
    TPMBinaryDecisionMapping,
    TPMCategoricalDecisionMapping,
    TPMCompilerFidelityEvidence,
    TPMContinuousDecisionMapping,
    TPMEmbeddingParityEvidence,
    attach_conditional_mio_pair,
    build_embedding_parity_evidence,
    evaluate_conditional_mio_pair,
    inspect_conditional_mio_attachment,
    load_conditional_pair,
    load_tpm_artifact,
)

REPOSITORY_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "tests/fixtures/tpm").is_dir()
    ),
    None,
)
if REPOSITORY_ROOT is None:
    raise FileNotFoundError(
        "Run this notebook from the OEL repository or one of its subdirectories."
    )
FIXTURE_ROOT = REPOSITORY_ROOT / "tests/fixtures/tpm"
SOURCE_PATH = FIXTURE_ROOT / "conditional_mio_embedding_v1.json"
ARTIFACT_PATH = FIXTURE_ROOT / "tpm_artifact_conditional_mio_v3.json"
MAX_PAIR_PATH = FIXTURE_ROOT / "conditional_pair_max_v1.json"
PWL_PAIR_PATH = FIXTURE_ROOT / "conditional_pair_pwl_v1.json"
EVIDENCE_PATH = FIXTURE_ROOT / "conditional_mio_evidence_v1.json"

## 1. Reload reviewed evidence

The source rows deliberately contain nonuniform evidence for the continuous decision. This matters because the learned marginal $p_X(x)$ can vary with $x$: the denominator cannot be replaced by a uniform constant merely because uniform sampling may have been intended.

The persisted schema-v3 artifact retains the U-Bounds declarations and current-SPFlow model. Here, `artifact_schema` describes that fitted artifact; it is not the conditional-pair schema version. Both conditional pairs identify the same fitted source and event meaning but use different recorded Sum encodings: `max` or `piecewise`. The two parity records and two fidelity records provide one record of each kind for each encoding.

Read the first displayed dictionary as an inventory and provenance check: it tells us which reviewed objects were loaded. It contains no model-selection verdict and does not compare the quality of max with PWL.

In [ ]:
source_fixture = json.loads(SOURCE_PATH.read_text(encoding="utf-8"))
artifact = load_tpm_artifact(ARTIFACT_PATH)
max_pair = load_conditional_pair(MAX_PAIR_PATH)
pwl_pair = load_conditional_pair(PWL_PAIR_PATH)
evidence_bundle = json.loads(EVIDENCE_PATH.read_text(encoding="utf-8"))

loaded_parity_records = tuple(
    TPMEmbeddingParityEvidence.from_dict(record)
    for record in evidence_bundle["embedding_parity_records"]
)
loaded_fidelity_records = tuple(
    TPMCompilerFidelityEvidence.from_dict(record)
    for record in evidence_bundle["compiler_fidelity_records"]
)
assignments_by_case = {
    case["case_name"]: dict(case["values"])
    for case in source_fixture["representative_assignments"]
}

reviewed_inputs = {
    "artifact_schema": artifact.schema_version,
    "artifact_id": artifact.artifact_id,
    "pair_modes": (max_pair.encoding.mode, pwl_pair.encoding.mode),
    "parity_record_count": len(loaded_parity_records),
    "fidelity_record_count": len(loaded_fidelity_records),
}
reviewed_inputs

## 2. Attach all supported mapping forms

Each diagnostic model belongs to the caller and contains: a continuous physical decision `x`, a binary `activation`, and three caller-owned binary indicators ordered as `economy`, `standard`, and `priority`. OEL references those variables rather than replacing them. It links physical $x\in[40,160]$ to the persisted model coordinate, interprets the binary states through their declared ordering, and adds exactly one constraint for the categorical indicators.

We fix just one representative assignment for each encoding. This is a solver-parity diagnostic, not a Cartesian query table. Direct evaluation executes the persisted compiled circuit without Pyomo; attached evaluation reads the roots produced after the identical circuit is attached and solved by HiGHS.

In the displayed summary, `direct_log_risk` and `attached_log_risk` should agree within the pair's $10^{-6}$ tolerance. `accepted=True` means that this identical-semantics parity check, solver status, and attachment-local residual checks passed. It does not mean that the compiled circuit matches native SPFlow within $10^{-6}$.

The max demonstration uses one boundary assignment while the PWL demonstration uses a different interior assignment. Their displayed risk values therefore must not be read as a head-to-head accuracy comparison. The later persisted evidence compares both encodings over the same representative cases.

Pyomo may report repeatedly that consecutive PWL segments have nearly equal slopes. That notice identifies almost-collinear neighboring segments; it is not by itself a failed solve. To keep the demonstration readable, the notebook captures only that exact notice, counts it, and reports one summary at the end. Any other construction output remains visible. The acceptance decision still comes from solver status, root agreement, and the recorded constraint-residual tolerance.

In [ ]:
CATEGORIES = ("economy", "standard", "priority")
NEAR_EQUAL_SLOPE_NOTICE = (
    "has detected slopes of consecutive piecewise segments to be within"
)
piecewise_notice_count = 0


def _attach_with_compact_piecewise_notices(*args, **kwargs):
    global piecewise_notice_count
    captured = StringIO()
    with redirect_stdout(captured):
        attachment = attach_conditional_mio_pair(*args, **kwargs)
    lines = captured.getvalue().splitlines(keepends=True)
    piecewise_notice_count += sum(
        NEAR_EQUAL_SLOPE_NOTICE in line for line in lines
    )
    other_output = "".join(
        line for line in lines if NEAR_EQUAL_SLOPE_NOTICE not in line
    )
    if other_output:
        print(other_output, end="")
    return attachment


def fixed_assignment_run(pair, case_name, namespace):
    assignments = assignments_by_case[case_name]
    model = pyo.ConcreteModel()
    model.x = pyo.Var(bounds=(40.0, 160.0))
    model.activation = pyo.Var(domain=pyo.Binary)
    model.service_tier = pyo.Var(CATEGORIES, domain=pyo.Binary)

    attachment = _attach_with_compact_piecewise_notices(
        pair,
        model,
        mappings=(
            TPMContinuousDecisionMapping("x", model.x),
            TPMBinaryDecisionMapping("activation", model.activation),
            TPMCategoricalDecisionMapping(
                "service_tier",
                tuple((label, model.service_tier[label]) for label in CATEGORIES),
            ),
        ),
        namespace=namespace,
    )

    model.x.fix(float(assignments["x"]))
    model.activation.fix(int(assignments["activation"] == "on"))
    for label in CATEGORIES:
        model.service_tier[label].fix(int(label == assignments["service_tier"]))

    direct = evaluate_conditional_mio_pair(
        pair, declared_domain_assignments=assignments
    )
    solver = pyo.SolverFactory("highs")
    solver.options["threads"] = 1
    result = solver.solve(model)
    attached = inspect_conditional_mio_attachment(
        attachment,
        solver_metadata={
            "solver_name": "highs",
            "solver_version": highspy.Highs().version(),
            "status": str(result.solver.status),
            "termination_condition": str(result.solver.termination_condition),
        },
    )
    parity = build_embedding_parity_evidence(
        pair,
        case_results={case_name: (direct, attached)},
        provenance={"consumer": "conditional-mio-walkthrough"},
    )
    return {
        "model": model,
        "attachment": attachment,
        "direct": direct,
        "attached": attached,
        "parity": parity,
    }

In [ ]:
max_run = fixed_assignment_run(max_pair, "lower-boundary", "max_risk")
pwl_run = fixed_assignment_run(pwl_pair, "interior", "pwl_risk")
max_parity = max_run["parity"]
pwl_parity = pwl_run["parity"]

fixed_assignment_summary = {
    "max": {
        "case": "lower-boundary",
        "accepted": max_parity.accepted,
        "direct_log_risk": max_run["direct"].conditional_log_risk,
        "attached_log_risk": max_run["attached"].conditional_log_risk,
    },
    "piecewise": {
        "case": "interior",
        "accepted": pwl_parity.accepted,
        "direct_log_risk": pwl_run["direct"].conditional_log_risk,
        "attached_log_risk": pwl_run["attached"].conditional_log_risk,
    },
}
fixed_assignment_summary

## 3. Keep two comparisons separate

**Embedding/solver parity** compares a Pyomo/HiGHS attachment with direct evaluation of the *identical compiled max or PWL semantics*. For a reported quantity $q$, the relevant difference is $q_{\mathrm{attached}}-q_{\mathrm{direct}}$. Root and log-risk errors plus attachment-local residuals are acceptance-gated at the persisted $10^{-6}$ tolerance. A failure here means the symbolic embedding or solve did not reproduce its own circuit. Therefore, `accepted` and `maximum_log_risk_error` describe implementation correctness only.

**Native/compiler fidelity** compares direct compiled values with native current-SPFlow numerator and denominator queries. Its signed convention is

$$\Delta_{\mathrm{fidelity}}=q_{\mathrm{compiled}}-q_{\mathrm{native}}.$$

A positive conditional-log-risk difference means the compiled approximation reports a larger log-risk than native SPFlow at that assignment; a negative difference means it reports a smaller one. The displayed tuples follow the reviewed case order: lower boundary, interior, then upper boundary. These discrepancies measure compiler approximation and are retained as scientific evidence with `numerical_gate='none'`. Max and PWL may disagree with native SPFlow while still passing embedding parity. This feature deliberately does not select the more favorable encoding or invent an adequacy threshold.

In [ ]:
embedding_solver_summary = tuple(
    {
        "mode": record.encoding["mode"],
        "accepted": record.accepted,
        "maximum_log_risk_error": max(
            case["absolute_errors"]["conditional_log_risk"]
            for case in record.cases
        ),
    }
    for record in loaded_parity_records
)
native_compiler_summary = tuple(
    {
        "mode": record.encoding["mode"],
        "numerical_gate": record.metric_definitions["numerical_gate"],
        "conditional_log_risk_differences": tuple(
            case["differences"]["conditional_log_risk"]
            for case in record.cases
        ),
    }
    for record in loaded_fidelity_records
)
{
    "embedding_solver_parity": embedding_solver_summary,
    "native_compiler_fidelity": native_compiler_summary,
}

### Visualizing the two evidence layers

The left panel visualizes embedding/solver correctness. Each point is the largest absolute conditional-log-risk difference between attached Pyomo evaluation and direct evaluation of the identical compiled pair. The dashed line is the persisted acceptance tolerance; points below it pass this operational check. Exact zero would be shown at a machine-precision plotting floor so that it remains visible on the logarithmic axis.

The right panel visualizes compiler approximation separately. Each bar is compiled minus native conditional log-risk for the same reviewed assignment. Zero means agreement with native SPFlow; positive and negative bars indicate larger and smaller compiled log-risk, respectively. There is intentionally no acceptance line in this panel because Feature 016 defines no compiler-adequacy threshold.

In [ ]:
parity_tolerances = {
    float(
        record.numerical_policy["tolerances"][
            "embedding_parity_absolute_tolerance"
        ]
    )
    for record in loaded_parity_records
}
if len(parity_tolerances) != 1:
    raise ValueError("reviewed parity records disagree on their tolerance")
parity_tolerance = parity_tolerances.pop()
mode_labels = {"max": "Max", "piecewise": "PWL"}
mode_colors = {"max": "#2563eb", "piecewise": "#ea580c"}
parity_modes = [record.encoding["mode"] for record in loaded_parity_records]
parity_errors = [
    max(
        case["absolute_errors"]["conditional_log_risk"]
        for case in record.cases
    )
    for record in loaded_parity_records
]
plot_floor = math.ulp(1.0)
plotted_parity_errors = [max(error, plot_floor) for error in parity_errors]

case_names = [case["case_name"] for case in loaded_fidelity_records[0].cases]
if any(
    [case["case_name"] for case in record.cases] != case_names
    for record in loaded_fidelity_records[1:]
):
    raise ValueError("reviewed fidelity records disagree on case ordering")

evidence_figure, (parity_axis, fidelity_axis) = plt.subplots(
    1, 2, figsize=(12, 4.5), constrained_layout=True
)
for mode, error in zip(parity_modes, plotted_parity_errors, strict=True):
    parity_axis.scatter(
        mode_labels[mode], error, s=95, color=mode_colors[mode], zorder=3
    )
parity_axis.axhline(
    parity_tolerance, color="#b91c1c", linestyle="--", label="Acceptance tolerance"
)
parity_axis.set_yscale("log")
parity_axis.set_ylim(
    min(plotted_parity_errors) / 3.0, parity_tolerance * 10.0
)
parity_axis.set_title("Embedding/solver parity")
parity_axis.set_ylabel("Maximum |attached − direct| log-risk error")
parity_axis.grid(axis="y", alpha=0.25)
parity_axis.legend(loc="upper left")

case_positions = list(range(len(case_names)))
bar_width = 0.34
for index, record in enumerate(loaded_fidelity_records):
    mode = record.encoding["mode"]
    offset = (index - (len(loaded_fidelity_records) - 1) / 2.0) * bar_width
    fidelity_axis.bar(
        [position + offset for position in case_positions],
        [
            case["differences"]["conditional_log_risk"]
            for case in record.cases
        ],
        width=bar_width,
        label=mode_labels[mode],
        color=mode_colors[mode],
    )
fidelity_axis.axhline(0.0, color="#334155", linewidth=1.0)
fidelity_axis.set_xticks(case_positions, case_names, rotation=15)
fidelity_axis.set_title("Native/compiler fidelity")
fidelity_axis.set_ylabel("Compiled − native conditional log-risk")
fidelity_axis.grid(axis="y", alpha=0.25)
fidelity_axis.legend()
evidence_figure.suptitle("Two distinct conditional-MIO evidence layers")
evidence_figure

## 4. Compose an unsolved caller policy

The final model demonstrates the intended reuse boundary. The physical decision is the affine expression `base_order + 2 * reserve`; the remaining mappings use a caller binary and ordered one-hot indicators. After attachment, the caller adds an unrelated deterministic constraint, its own $\alpha$, the chance constraint

$$z_{\mathrm{num}}(x)-z_{\mathrm{den}}(x) \leq \log \alpha,$$

and its own objective. OEL does not choose the threshold, add that constraint, or invoke the solver. This notebook likewise leaves the policy unsolved: it demonstrates composition, not a benchmark recommendation.

Read the final dictionary as an ownership audit. `attachment_namespace` identifies the single OEL-owned block. `caller_components` lists objects added by the surrounding model rather than by OEL. `policy_unsolved=True` confirms that no policy solve was requested. `near_equal_pwl_slope_notices` reports how many duplicate Pyomo notices were compacted; it is diagnostic context, not a failure count. `conditional_demo_complete=True` means the fixture loading and fixed-assignment parity demonstrations completed; it is not a statement that this policy is feasible, optimal, calibrated, or scientifically adequate.

In [ ]:
policy_model = pyo.ConcreteModel()
policy_model.base_order = pyo.Var(bounds=(40.0, 150.0))
policy_model.reserve = pyo.Var(bounds=(0.0, 5.0))
policy_model.activation = pyo.Var(domain=pyo.Binary)
policy_model.service_tier = pyo.Var(CATEGORIES, domain=pyo.Binary)

policy_attachment = _attach_with_compact_piecewise_notices(
    max_pair,
    policy_model,
    mappings=(
        TPMContinuousDecisionMapping(
            "x", policy_model.base_order + 2.0 * policy_model.reserve
        ),
        TPMBinaryDecisionMapping("activation", policy_model.activation),
        TPMCategoricalDecisionMapping(
            "service_tier",
            tuple(
                (label, policy_model.service_tier[label])
                for label in CATEGORIES
            ),
        ),
    ),
    namespace="policy_risk",
)

policy_model.caller_deterministic_constraint = pyo.Constraint(
    expr=policy_model.base_order + policy_model.reserve <= 150.0
)
policy_model.caller_alpha = pyo.Param(initialize=0.6, mutable=True)
policy_model.caller_chance_constraint = pyo.Constraint(
    expr=policy_attachment.conditional_log_risk
    <= math.log(pyo.value(policy_model.caller_alpha))
)
policy_model.caller_objective = pyo.Objective(
    expr=policy_model.base_order
    + 3.0 * policy_model.reserve
    + 2.0 * policy_model.activation
)

policy_unsolved = len(policy_model.solutions) == 0
conditional_demo_complete = (
    max_parity.accepted
    and pwl_parity.accepted
    and all(record.accepted for record in loaded_parity_records)
    and policy_unsolved
)
{
    "attachment_namespace": policy_attachment.namespace,
    "caller_components": (
        "caller_objective",
        "caller_deterministic_constraint",
        "caller_alpha",
        "caller_chance_constraint",
    ),
    "policy_unsolved": policy_unsolved,
    "near_equal_pwl_slope_notices": piecewise_notice_count,
    "conditional_demo_complete": conditional_demo_complete,
}